In [ ]:
import os
import re
import sys
import random
import shutil
import zipfile
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
from scipy.stats import ks_2samp, wasserstein_distance
from scipy.spatial.distance import jensenshannon
from IPython.display import display

BRANCH = "virtual_grid-gnn"
REPOSITORY_URL = "https://github.com/DaryaBor/FHCAL_GAN.git"
MODEL_NAME = "physics_virtual25_dilated_gnn_layers7_10"
DATA_FILENAME = "fhcal_data_muons_81619.npz"
DATA_PATH = None
CHECKPOINT_PATH = None
SEED = 42
N_SAMPLES = 10000
BATCH_SIZE = 64
NEAR_ZERO = 1e-6
NOISE_SAMPLES = 1000
NOISE_CONDITIONS = 30
NOISE_REPEATS = 100
DILATION_RATES = (1, 2, 4)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)


In [ ]:
working = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
current = Path.cwd()
if (current / "pipeline/generators.py").is_file():
    project_dir = current
    repo_dir = current.parent
elif (current / "GAN_project/pipeline/generators.py").is_file():
    repo_dir = current
    project_dir = repo_dir / "GAN_project"
else:
    repo_dir = working / "FHCAL_GAN"
    if not (repo_dir / ".git").is_dir():
        if repo_dir.exists():
            raise RuntimeError(f"Каталог уже существует без Git: {repo_dir}")
        subprocess.run([
            "git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
            "--filter=blob:none", "--sparse", REPOSITORY_URL, str(repo_dir)
        ], check=True)
        subprocess.run([
            "git", "-C", str(repo_dir), "sparse-checkout", "set", "--no-cone",
            "/GAN_project/pipeline/", "/GAN_project/config/",
            "/GAN_project/data/fhcal_data_muons_81619.zip"
        ], check=True)
    project_dir = repo_dir / "GAN_project"
active_branch = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "--abbrev-ref", "HEAD"], text=True
).strip()
if active_branch != BRANCH:
    raise RuntimeError(f"В {repo_dir} выбрана ветка {active_branch}; требуется {BRANCH}")
revision = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
).strip()
os.chdir(project_dir)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
from pipeline import data as physics_data
from pipeline.gan import GAN
from pipeline.generators import CaloganPhysicsGenerator3D
from pipeline.discriminators import CaloganPhysicsDiscriminator3D
if not hasattr(physics_data, "fhcal_geometry_report_19"):
    raise RuntimeError("Код ветки устарел: отсутствует fhcal_geometry_report_19")
print("Проект:", project_dir)
print("Ветка:", active_branch)
print("Коммит:", revision)


In [ ]:
input_roots = [Path("/kaggle/input")] if Path("/kaggle/input").is_dir() else []

def choose_file(explicit, candidates, label):
    if explicit is not None:
        path = Path(explicit).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"{label}: {path}")
        return path
    paths = sorted({p.resolve() for p in candidates if p.is_file()})
    if len(paths) != 1:
        listing = "\n".join(str(p) for p in paths) or "нет кандидатов"
        raise RuntimeError(f"Укажи {label} в первой ячейке. Кандидаты:\n{listing}")
    return paths[0]

if DATA_PATH is None:
    mounted = [p for root in input_roots for p in root.rglob(DATA_FILENAME)]
    local_data = project_dir / "data" / DATA_FILENAME
    if mounted:
        DATA_PATH = choose_file(None, mounted, "DATA_PATH")
    elif local_data.is_file():
        DATA_PATH = local_data.resolve()
    else:
        archive_path = project_dir / "data" / f"{Path(DATA_FILENAME).stem}.zip"
        if not archive_path.is_file():
            raise FileNotFoundError(f"Не найден {DATA_FILENAME}; укажи DATA_PATH в первой ячейке")
        local_data.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            members = [name for name in archive.namelist() if Path(name).name == DATA_FILENAME]
            if len(members) != 1:
                raise RuntimeError(f"В архиве найдено {len(members)} файлов {DATA_FILENAME}")
            with archive.open(members[0]) as source, local_data.open("wb") as target:
                shutil.copyfileobj(source, target)
        DATA_PATH = local_data.resolve()
else:
    DATA_PATH = choose_file(DATA_PATH, [], "DATA_PATH")

if CHECKPOINT_PATH is None:
    local_checkpoint = project_dir / "experiments" / MODEL_NAME / "training_checkpoint"
    mounted_checkpoints = [p for root in input_roots for p in root.rglob("training_checkpoint*") if p.is_file()]
    matching = [p for p in mounted_checkpoints if MODEL_NAME in str(p)]
    if local_checkpoint.is_file():
        CHECKPOINT_PATH = local_checkpoint.resolve()
    else:
        CHECKPOINT_PATH = choose_file(None, matching or mounted_checkpoints, "CHECKPOINT_PATH")
else:
    CHECKPOINT_PATH = choose_file(CHECKPOINT_PATH, [], "CHECKPOINT_PATH")
print("NPZ:", DATA_PATH)
print("Чекпойнт:", CHECKPOINT_PATH)


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if "gan" not in checkpoint or not {"generator", "discriminator"}.issubset(checkpoint["gan"]):
    raise ValueError("Ожидается training_checkpoint с gan.generator и gan.discriminator")
generator_state = checkpoint["gan"]["generator"]
critic_state = checkpoint["gan"]["discriminator"]
required = {"input.0.weight", "block1.0.weight", "block2.0.weight", "block3.0.weight",
            "condition.0.weight", "condition_mean", "condition_scale", "pdg_codes"}
if not required.issubset(generator_state):
    raise ValueError("Чекпойнт не соответствует генератору с полной картой и dilation")
condition_dim = generator_state["condition.0.weight"].shape[1]
generator_hidden = generator_state["condition.0.weight"].shape[0]
noise_dim = generator_state["input.0.weight"].shape[1] - generator_hidden - 6
critic_hidden = critic_state["condition.0.weight"].shape[0]
block_ids = sorted({int(key.split(".")[1]) for key in critic_state if re.match(r"blocks\.\d+\.", key)})
if not block_ids or block_ids != list(range(len(block_ids))):
    raise ValueError("Неподдерживаемая структура GNN-критика")
if condition_dim != 6 + generator_state["pdg_codes"].numel() or noise_dim <= 0:
    raise ValueError("Несогласованные размерности условий и шума")
condition_state = {
    "mean": generator_state["condition_mean"],
    "scale": generator_state["condition_scale"],
    "pdg_codes": generator_state["pdg_codes"].tolist(),
}
print("Следующая эпоха:", checkpoint.get("epoch"))
print("Размерность шума:", noise_dim)
print("Размерность условий:", condition_dim)
print("PDG:", condition_state["pdg_codes"])
print("Dilation:", DILATION_RATES)


In [ ]:
geometry, module_map = physics_data.fhcal_geometry_report_19()
expected_geometry = geometry.state_dict()
for name, expected in expected_geometry.items():
    for label, state in (("генератор", generator_state), ("критик", critic_state)):
        key = f"geometry.{name}"
        if key not in state or state[key].shape != expected.shape:
            raise ValueError(f"Геометрия чекпойнта не совпадает: {label}, {name}")
        if expected.is_floating_point():
            equal = torch.allclose(state[key], expected, rtol=1e-5, atol=1e-6)
        else:
            equal = torch.equal(state[key], expected)
        if not equal:
            raise ValueError(f"Геометрия чекпойнта не совпадает: {label}, {name}")
val_dataset = physics_data.get_physics_dataset(
    str(DATA_PATH), train=False, geometry=geometry, module_map=module_map
)
for name in ("mean", "scale"):
    if not torch.allclose(val_dataset.condition_state[name], condition_state[name], rtol=1e-5, atol=1e-6):
        raise ValueError(f"Статистика условий {name} не совпадает с обучением; проверь NPZ")
if val_dataset.condition_state["pdg_codes"] != condition_state["pdg_codes"]:
    raise ValueError("Словарь PDG в NPZ отличается от чекпойнта")
if val_dataset.condition_dim != condition_dim or len(val_dataset) == 0:
    raise ValueError("Некорректная валидационная выборка")
module_ids = geometry.module_ids.cpu().numpy().copy()
valid_mask = geometry.layer_mask.cpu().numpy().astype(bool)
id_to_index = {int(mid): i for i, mid in enumerate(module_ids)}
print("Validation:", len(val_dataset))
print("Виртуальная карта:", (geometry.height, geometry.width))
print("Допустимых модульных слоёв:", int(valid_mask.sum()))


In [ ]:
generator = CaloganPhysicsGenerator3D(
    geometry, condition_dim, noise_dim=noise_dim, hidden=generator_hidden,
    condition_state=condition_state, dilation_rates=DILATION_RATES
)
critic = CaloganPhysicsDiscriminator3D(
    geometry, condition_dim, hidden=critic_hidden, depth=len(block_ids)
)
gan = GAN(generator, critic, noise_generator=lambda n: torch.randn(n, noise_dim))
gan.load_state_dict(checkpoint["gan"])
gan.to(device)
gan.eval()
print("Модель загружена")


In [ ]:
def get_real_fake_samples(gan, dataset, n_samples=N_SAMPLES, batch_size=BATCH_SIZE, seed=SEED):
    if n_samples < 1 or batch_size < 1:
        raise ValueError("Размеры выборки и батча должны быть положительными")
    indices = np.random.default_rng(seed).choice(len(dataset), size=min(n_samples, len(dataset)), replace=False)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(dataset, indices.tolist()), batch_size=batch_size,
        shuffle=False, num_workers=0
    )
    noise_rng = torch.Generator(device=device).manual_seed(seed)
    real_batches, fake_batches, condition_batches = [], [], []
    real_scores, fake_scores = [], []
    with torch.inference_mode():
        for real_batch, condition_batch in loader:
            c = condition_batch.to(device)
            z = torch.randn(len(c), noise_dim, device=device, generator=noise_rng)
            generated_batch = gan.generator(z, c)
            real_scores.append(gan.discriminator(real_batch.to(device), c).cpu().numpy())
            fake_scores.append(gan.discriminator(generated_batch, c).cpu().numpy())
            real_batches.append(real_batch.numpy())
            fake_batches.append(generated_batch.cpu().numpy())
            condition_batches.append(condition_batch.numpy())
    return tuple(np.concatenate(items) for items in (
        real_batches, fake_batches, condition_batches, real_scores, fake_scores
    )) + (indices,)


In [ ]:
real, fake, sample_conditions, real_scores, fake_scores, sample_indices = get_real_fake_samples(gan, val_dataset)
if real.shape != fake.shape or real.shape[1:] != (len(module_ids), geometry.max_layers):
    raise ValueError(f"Некорректные формы REAL/FAKE: {real.shape}, {fake.shape}")
for label, values in (("REAL", real), ("FAKE", fake)):
    if not np.isfinite(values).all() or np.any(values < 0):
        raise ValueError(f"{label}: отрицательные или нечисловые энергии")
    if np.count_nonzero(values[:, ~valid_mask]):
        raise ValueError(f"{label}: энергия в отсутствующих слоях")
real_total = real.sum(axis=(1, 2))
fake_total = fake.sum(axis=(1, 2))
real_layers = real.sum(axis=1)
fake_layers = fake.sum(axis=1)
real_modules = real.sum(axis=2)
fake_modules = fake.sum(axis=2)
print("REAL:", real.shape)
print("FAKE:", fake.shape)
print("Условия:", sample_conditions.shape)


In [ ]:
def common_bins(a, b, count=50):
    values = np.concatenate((np.asarray(a).reshape(-1), np.asarray(b).reshape(-1)))
    if values.size == 0 or not np.isfinite(values).all():
        raise ValueError("Пустые или нечисловые значения для гистограммы")
    lo, hi = float(values.min()), float(values.max())
    if lo == hi:
        span = max(abs(lo) * 1e-3, 1e-6)
        lo, hi = lo - span, hi + span
    return np.linspace(lo, hi, count + 1)

def distribution_metrics(a, b):
    a, b = np.asarray(a).reshape(-1), np.asarray(b).reshape(-1)
    if not len(a) or not len(b):
        return {"JS": np.nan, "KS": np.nan, "KS p-value": np.nan, "W1": np.nan}
    bins = common_bins(a, b)
    pa = np.histogram(a, bins=bins)[0].astype(float)
    pb = np.histogram(b, bins=bins)[0].astype(float)
    pa, pb = pa / pa.sum(), pb / pb.sum()
    ks, pvalue = ks_2samp(a, b)
    return {"JS": float(jensenshannon(pa, pb, base=2) ** 2), "KS": float(ks),
            "KS p-value": float(pvalue), "W1": float(wasserstein_distance(a, b))}


In [ ]:
bins = common_bins(real_total, fake_total, 80)
plt.figure(figsize=(9, 5))
plt.hist(real_total, bins=bins, density=True, alpha=.5, label="REAL")
plt.hist(fake_total, bins=bins, density=True, alpha=.5, label="FAKE")
plt.xlabel("Полная энергия, единицы NPZ")
plt.ylabel("Плотность вероятности")
plt.title("Полная энергия события")
plt.legend()
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()


In [ ]:
quantile_levels = [.01, .05, .25, .5, .75, .95, .99]
stats = pd.DataFrame({
    "REAL": [real_total.mean(), real_total.std(), *np.quantile(real_total, quantile_levels)],
    "FAKE": [fake_total.mean(), fake_total.std(), *np.quantile(fake_total, quantile_levels)],
}, index=["Mean", "Std", "Q01", "Q05", "Q25", "Q50", "Q75", "Q95", "Q99"])
display(stats)
display(pd.DataFrame([distribution_metrics(real_total, fake_total)], index=["Полная энергия"]))


In [ ]:
real_fractions = real_layers / np.maximum(real_total[:, None], 1e-12)
fake_fractions = fake_layers / np.maximum(fake_total[:, None], 1e-12)
layers = np.arange(1, geometry.max_layers + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, r, f, title, ylabel in (
    (axes[0], real_layers, fake_layers, "Абсолютный продольный профиль", "Средняя энергия, единицы NPZ"),
    (axes[1], real_fractions, fake_fractions, "Средние доли энергии по слоям", "Доля энергии"),
):
    ax.plot(layers, r.mean(axis=0), "o-", label="REAL")
    ax.plot(layers, f.mean(axis=0), "o-", label="FAKE")
    ax.set(xlabel="Номер слоя", ylabel=ylabel, title=title, xticks=layers)
    ax.grid(alpha=.3)
    ax.legend()
plt.tight_layout()
plt.show()
print("MAE средних долей:", np.abs(real_fractions.mean(0) - fake_fractions.mean(0)).mean())


In [ ]:
layer_metrics = pd.DataFrame([
    {"Слой": i + 1, "Модулей с этим слоем": int(valid_mask[:, i].sum()),
     "REAL mean": real_layers[:, i].mean(), "FAKE mean": fake_layers[:, i].mean(),
     **distribution_metrics(real_layers[:, i], fake_layers[:, i])}
    for i in range(geometry.max_layers)
])
display(layer_metrics)


In [ ]:
small = np.array([m.width == m.height == 15 for m in geometry.modules])
x_positions = np.array([m.x for m in geometry.modules])
center_x = (x_positions[small].min() + x_positions[small].max()) / 2
regions = {"Центр, 15×15 см": small,
           "Лево, 20×20 см": (~small) & (x_positions < center_x),
           "Право, 20×20 см": (~small) & (x_positions >= center_x)}
fig, axes = plt.subplots(2, 3, figsize=(17, 8), squeeze=False)
regional_rows = []
for column, (name, mask) in enumerate(regions.items()):
    available = valid_mask[mask].any(axis=0)
    r = real[:, mask, :].sum(axis=1)[:, available]
    f = fake[:, mask, :].sum(axis=1)[:, available]
    labels = np.flatnonzero(available) + 1
    rfrac = r / np.maximum(r.sum(1, keepdims=True), 1e-12)
    ffrac = f / np.maximum(f.sum(1, keepdims=True), 1e-12)
    for row, ra, fa, ylabel in ((0, r, f, "Средняя энергия"), (1, rfrac, ffrac, "Средняя доля")):
        axes[row, column].plot(labels, ra.mean(0), "o-", label="REAL")
        axes[row, column].plot(labels, fa.mean(0), "o-", label="FAKE")
        axes[row, column].set(title=name, xlabel="Слой", ylabel=ylabel, xticks=labels)
        axes[row, column].grid(alpha=.3)
        axes[row, column].legend()
    for index, layer in enumerate(labels):
        regional_rows.append({"Область": name, "Слой": int(layer), **distribution_metrics(r[:, index], f[:, index])})
plt.tight_layout()
plt.show()
regional_metrics = pd.DataFrame(regional_rows)
display(regional_metrics)


In [ ]:
module_rows = []
for index, mid in enumerate(module_ids):
    r, f = real_modules[:, index], fake_modules[:, index]
    positive = distribution_metrics(r[r > 0], f[f > 0])
    module_rows.append({
        "Module ID": int(mid), "Слоёв": int(valid_mask[index].sum()),
        "REAL mean": r.mean(), "FAKE mean": f.mean(),
        "REAL zero": np.mean(r == 0), "FAKE zero": np.mean(f == 0),
        "REAL near zero": np.mean(r <= NEAR_ZERO), "FAKE near zero": np.mean(f <= NEAR_ZERO),
        "REAL positive count": int((r > 0).sum()), "FAKE positive count": int((f > 0).sum()),
        **distribution_metrics(r, f),
        **{f"positive {key}": value for key, value in positive.items()},
    })
module_metrics = pd.DataFrame(module_rows).set_index("Module ID")
display(module_metrics)


In [ ]:
focus_ids = [mid for mid in (8, 16, 19) if mid in id_to_index]
fig, axes = plt.subplots(1, len(focus_ids), figsize=(5 * len(focus_ids), 4), squeeze=False)
for ax, mid in zip(axes[0], focus_ids):
    index = id_to_index[mid]
    r, f = real_modules[:, index], fake_modules[:, index]
    bins = common_bins(r, f, 50)
    ax.hist(r, bins=bins, density=True, alpha=.5, label="REAL")
    ax.hist(f, bins=bins, density=True, alpha=.5, label="FAKE")
    ax.set(title=f"Модуль №{mid}", xlabel="Энергия модуля, единицы NPZ", ylabel="Плотность")
    ax.grid(alpha=.3)
    ax.legend()
plt.tight_layout()
plt.show()
display(module_metrics.loc[focus_ids])


In [ ]:
positions = np.arange(len(module_ids))
plt.figure(figsize=(18, 5))
plt.bar(positions - .2, (real_modules == 0).mean(0), width=.4, label="REAL")
plt.bar(positions + .2, (fake_modules == 0).mean(0), width=.4, label="FAKE")
plt.xticks(positions, module_ids, fontsize=8)
plt.xlabel("Реальный ID модуля")
plt.ylabel("Доля событий с нулевой энергией модуля")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(axis="y", alpha=.3)
plt.tight_layout()
plt.show()


In [ ]:
activity = pd.DataFrame([
    {"Порог, единицы NPZ": threshold,
     "REAL, доля активных модульных слоёв": np.mean(real[:, valid_mask] > threshold),
     "FAKE, доля активных модульных слоёв": np.mean(fake[:, valid_mask] > threshold),
     "REAL, доля активных модулей": np.mean(real_modules > threshold),
     "FAKE, доля активных модулей": np.mean(fake_modules > threshold)}
    for threshold in (0., NEAR_ZERO, .001, .01, .1, 1.)
])
display(activity)


In [ ]:
means = (real_modules.mean(0), fake_modules.mean(0))
normalization = Normalize(vmin=min(a.min() for a in means), vmax=max(a.max() for a in means))
cmap = plt.get_cmap("viridis")
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
for ax, values, label in zip(axes, means, ("REAL", "FAKE")):
    for module, value in zip(geometry.modules, values):
        ax.add_patch(Rectangle((module.x - module.width / 2, module.y - module.height / 2),
                               module.width, module.height, facecolor=cmap(normalization(value)), edgecolor="black"))
        ax.text(module.x, module.y, str(module.module_id), ha="center", va="center", fontsize=8)
    ax.set_xlim(min(m.x - m.width / 2 for m in geometry.modules), max(m.x + m.width / 2 for m in geometry.modules))
    ax.set_ylim(min(m.y - m.height / 2 for m in geometry.modules), max(m.y + m.height / 2 for m in geometry.modules))
    ax.set_aspect("equal")
    ax.set(title=f"{label}: средняя энергия физических модулей", xlabel="X, см", ylabel="Y, см")
fig.colorbar(plt.cm.ScalarMappable(norm=normalization, cmap=cmap), ax=list(axes), label="Средняя энергия, единицы NPZ")
plt.show()


In [ ]:
bins = common_bins(real_scores, fake_scores, 60)
plt.figure(figsize=(9, 4))
plt.hist(real_scores, bins=bins, density=True, alpha=.5, label="D(REAL,c)")
plt.hist(fake_scores, bins=bins, density=True, alpha=.5, label="D(FAKE,c)")
plt.xlabel("Оценка критика")
plt.ylabel("Плотность")
plt.legend()
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()
print("Средняя оценка REAL:", real_scores.mean())
print("Средняя оценка FAKE:", fake_scores.mean())


In [ ]:
unique_conditions, first_occurrence = np.unique(sample_conditions, axis=0, return_index=True)
example_positions = np.sort(first_occurrence)[:3]
example_rows = []
mean = condition_state["mean"].numpy()
scale = condition_state["scale"].numpy()
for position in example_positions:
    encoded = sample_conditions[position]
    raw = encoded[:6] * scale + mean
    example_rows.append({"Validation index": int(sample_indices[position]),
                         **dict(zip(("x0", "y0", "z0", "px", "py", "pz"), raw)),
                         "PDG": condition_state["pdg_codes"][int(encoded[6:].argmax())]})
display(pd.DataFrame(example_rows))
print("Уникальных подготовленных условий в выборке:", len(unique_conditions))


In [ ]:
def generate_at_condition(condition, n_samples, seed, batch_size=BATCH_SIZE):
    noise_rng = torch.Generator(device=device).manual_seed(seed)
    condition = torch.as_tensor(condition, dtype=torch.float32, device=device).reshape(1, -1)
    totals = []
    with torch.inference_mode():
        for start in range(0, n_samples, batch_size):
            count = min(batch_size, n_samples - start)
            c = condition.expand(count, -1)
            z = torch.randn(count, noise_dim, device=device, generator=noise_rng)
            generated_batch = gan.generator(z, c)
            totals.append(generated_batch.sum(dim=(1, 2)).cpu().numpy())
    return np.concatenate(totals)


In [ ]:
fig, axes = plt.subplots(1, len(example_positions), figsize=(5 * len(example_positions), 4), squeeze=False)
noise_rows = []
for ax, position in zip(axes[0], example_positions):
    values = generate_at_condition(sample_conditions[position], NOISE_SAMPLES, SEED + int(position))
    source_index = int(sample_indices[position])
    ax.hist(values, bins=40, density=True, alpha=.7)
    ax.axvline(real_total[position], color="red", linestyle="--", label="Одно REAL-событие")
    ax.set(title=f"Фиксированные условия, событие {source_index}", xlabel="Полная энергия, единицы NPZ", ylabel="Плотность")
    ax.grid(alpha=.3)
    ax.legend()
    noise_rows.append({"Validation index": source_index, "Mean FAKE": values.mean(), "Std FAKE": values.std(),
                       "Min": values.min(), "Max": values.max(),
                       **{f"Q{int(q * 100):02d}": value for q, value in zip((.05, .5, .95), np.quantile(values, (.05, .5, .95)))}})
plt.tight_layout()
plt.show()
display(pd.DataFrame(noise_rows))


In [ ]:
condition_count = min(NOISE_CONDITIONS, len(unique_conditions))
chosen = np.random.default_rng(SEED).choice(len(unique_conditions), size=condition_count, replace=False)
noise_energy_samples = np.stack([
    generate_at_condition(unique_conditions[index], NOISE_REPEATS, SEED + 1000 + int(index))
    for index in chosen
])
within_std = noise_energy_samples.std(axis=1)
condition_means = noise_energy_samples.mean(axis=1)
display(pd.DataFrame([{
    "Уникальных условий": condition_count,
    "Генераций на условие": NOISE_REPEATS,
    "Средний Std внутри условия": within_std.mean(),
    "Минимальный Std внутри условия": within_std.min(),
    "Максимальный Std внутри условия": within_std.max(),
    "Std средних между условиями": condition_means.std(),
    "Общий Std": noise_energy_samples.std(),
}]))
